In [0]:
%sql

CREATE OR REPLACE TABLE `03_gold_currency`.fato_cotacoes
USING DELTA
AS
  
-- Mínimas datas por moeda
WITH base_dados AS (
SELECT
  MIN(cot.dt_periodo) AS min_data,
  cot.dc_moeda
FROM `02_silver_currency`.cotacoes cot 
GROUP BY cot.dc_moeda
),

-- Gera todas as datas do período
datas_periodo AS (
SELECT
    dc_moeda,
    explode(
        sequence(
            min_data,
            current_date(), --Data atual
            interval 1 day
        )
    ) AS dt_periodo
FROM base_dados
),

-- Mescla as cotações atuais com as datas do período
dados_completos AS (
SELECT
    dp.dt_periodo,
    dp.dc_moeda,
    COALESCE(cot.vl_cotacao, NULL) AS vl_cotacao
FROM datas_periodo dp
LEFT JOIN `02_silver_currency`.cotacoes cot 
ON dp.dc_moeda = cot.dc_moeda 
    AND dp.dt_periodo = cot.dt_periodo
),

-- Preenche as lacunas com a última cotação de cada moeda
fato_cotacoes_final AS (
SELECT
    dt_periodo,
    dc_moeda,
    last_value(vl_cotacao, true)
    OVER (
        PARTITION BY dc_moeda
        ORDER BY dt_periodo
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS vl_cotacao
FROM dados_completos
)
SELECT * FROM fato_cotacoes_final
ORDER BY dt_periodo, dc_moeda

In [0]:
%sql
SELECT * FROM 03_gold_currency.fato_cotacoes
ORDER BY dt_periodo, dc_moeda